In [0]:
#O objetivo do projeto era simular um pipeline de engenharia de dados em produção. Embora o arquivo fosse pequeno, utilizei Spark porque ele é a tecnologia empregada em ambientes distribuídos e permite que o mesmo código escale para volumes muito maiores, sem mudanças significativas na lógica.
#Lendo primeiro arquivo.

df = spark.read.csv(
    "/Volumes/cafe_sales_2026/bronze/volume/dirty_cafe_sales.csv",
    header=True,
    inferSchema=True
)

# Visualiza as primeiras linhas pra conferir se leu certo
display(df) 

In [0]:
# Renomeia colunas substituindo espaços por underscore na primiera tabela. 1/2.
for col in df.columns:
    df = df.withColumnRenamed(col, col.replace(" ", "_"))

display(df)

In [0]:
#Utilizamos essa função para contar a quantidade de valores nulos em cada coluna do DataFrame, auxiliando na validação da qualidade dos dados antes das transformações.

from pyspark.sql.functions import col, when, sum

null_counts = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
#Utilizamos printSchema para visualizar o tipo de dados de cada coluna.
df.printSchema()

In [0]:
#transformando a primeira tabela em delta e salvando no bronze. 1/2.
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cafe_sales_2026.bronze.dirty_cafe_sales")

In [0]:
%sql
--Conferindo a contagem de registros da primeira tabela - 1/2.
select count(*) from cafe_sales_2026.bronze.dirty_cafe_sales

In [0]:
#Lendo Segundo arquivo - 2/2.
df = spark.read.csv(
    '/Volumes/cafe_sales_2026/bronze/volume/dirty_cafe_sales_new.csv',
    header=True,
    inferSchema=True
)

# Visualiza as primeiras linhas pra conferir se leu certo
display(df) 

In [0]:
# Renomeia colunas substituindo espaços por underscore na segunda tabela. 2/2.
for col in df.columns:
    df = df.withColumnRenamed(col, col.replace(" ", "_"))

display(df)

In [0]:
#Utilizamos printSchema para visualizar o tipo de dados de cada coluna.
df.printSchema()

In [0]:
#Utilizamos essa função para contar a quantidade de valores nulos em cada coluna do DataFrame, auxiliando na validação da qualidade dos dados antes das transformações.

from pyspark.sql.functions import col, when, sum

null_counts = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
#transformando a segunda tabela em delta e salvando no bronze. 2/2.
df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "cafe_sales_2026.bronze.dirty_cafe_sales"
    )

In [0]:
%sql
--Conferindo a contagem de registros da primeira/segunda tabela - 2/2.

SELECT COUNT(*)
FROM cafe_sales_2026.bronze.dirty_cafe_sales